# Phase 1 — Lesson 21: Graph Theory for Machine Learning
> Faithful study notes built directly from the lesson source. Concepts, code, diagrams, and exercises.

**Source:** ai-engineering-from-scratch -> Phase 1 (Math Foundations), Lesson 21: Graph Theory for Machine Learning
**Type:** Build | **Languages:** Python | **Time:** ~90 minutes
**Prerequisites:** Phase 1, Lessons 01-03



# What You'll Learn

---

## Learning Objectives

- Build a graph class with adjacency matrix/list representations and implement BFS and DFS traversals
- Compute the graph Laplacian and use its eigenvalues to detect connected components and cluster nodes
- Implement one round of GNN-style message passing as a normalized adjacency matrix multiplication
- Apply spectral clustering to partition a graph using the Fiedler vector


# What Is a Graph?

---

A graph G = (V, E) consists of vertices (nodes) V and edges E.

| Type | Example |
|------|---------|
| Undirected, unweighted | Facebook friendship |
| Directed, unweighted | Twitter follows |
| Undirected, weighted | Road map (distances) |
| Directed, weighted | Web links (PageRank) |


In [ ]:
import numpy as npimport matplotlib.pyplot as pltimport networkx as nx# Build a small social networkG = nx.Graph()G.add_edges_from([(0, 1), (0, 2), (1, 2), (1, 3), (2, 3), (3, 4)])A = nx.adjacency_matrix(G).toarray()print("Adjacency matrix:")print(A)plt.figure(figsize=(6, 5))nx.draw(G, with_labels=True, node_color='#2c3e50', node_size=500,        font_color='white', font_weight='bold', edge_color='#95a5a6', width=2)plt.title('Small Social Network Graph', fontsize=12, fontweight='bold')plt.tight_layout(); plt.show()

In [ ]:
from collections import dequedef bfs(graph, start):    visited = {start}    queue = deque([start])    order = []    while queue:        node = queue.popleft()        order.append(node)        for neighbor in graph.neighbors(node):            if neighbor not in visited:                visited.add(neighbor)                queue.append(neighbor)    return orderdef dfs(graph, start):    visited = set()    stack = [start]    order = []    while stack:        node = stack.pop()        if node not in visited:            visited.add(node)            order.append(node)            stack.extend(reversed(list(graph.neighbors(node))))    return orderprint(f"BFS from node 0: {bfs(G, 0)}")print(f"DFS from node 0: {dfs(G, 0)}")

# Graph Laplacian

---

L = D − A where D is the degree matrix.

Key properties:
- L is positive semi-definite
- Number of zero eigenvalues = number of connected components
- Smallest non-zero eigenvalue (Fiedler value) = algebraic connectivity
- Fiedler vector = best split for spectral clustering


In [ ]:
import numpy as npimport matplotlib.pyplot as plt# Build a graph with two clear clustersG = nx.Graph()G.add_edges_from([(i, (i + 1) % 5) for i in range(5)])  # Cluster 1: cycle of 5G.add_edges_from([(i, (i + 1) % 5 + 5) for i in range(5)])  # Cluster 2: cycle of 5G.add_edge(2, 7)  # Single bridge edgeA = nx.adjacency_matrix(G).toarray()D = np.diag(A.sum(axis=1))L = D - Aeigenvalues, eigenvectors = np.linalg.eigh(L)print(f"Eigenvalues: {np.round(eigenvalues, 4)}")print(f"Number of zero eigenvalues: {np.sum(np.abs(eigenvalues) < 1e-8)}")print(f"Fiedler value: {eigenvalues[1]:.4f}")# Spectral clustering using Fiedler vectorfiedler = eigenvectors[:, 1]labels = (fiedler > 0).astype(int)pos = nx.circular_layout(G)plt.figure(figsize=(6, 5))nx.draw(G, pos, node_color=labels, cmap='viridis', with_labels=True,          node_size=500, font_color='white', font_weight='bold')plt.title(f'Spectral Clustering: {len(np.unique(labels))} clusters', fontsize=12, fontweight='bold')plt.tight_layout(); plt.show()print(f"Cluster 0: {np.where(labels == 0)[0]}")print(f"Cluster 1: {np.where(labels == 1)[0]}")

# Message Passing (GNN Core)

---

Each node aggregates messages from its neighbors:

```
H⁽ᵏ⁺¹₎ = σ(A_norm · H⁽ᵏ⁾ · W)
```

One round = each node sees its immediate neighbors.


In [ ]:
import numpy as npnp.random.seed(42)# Node features (4 nodes, 2 features each)H = np.random.randn(4, 2)# Adjacency matrix (triangle: 0-1-2-0, plus node 3 connected to 0)A = np.array([[0, 1, 1, 1], [1, 0, 1, 0], [1, 1, 0, 0], [1, 0, 0, 0]], dtype=float)# Normalize adjacency (row-stochastic)row_sums = A.sum(axis=1, keepdims=True)row_sums[row_sums == 0] = 1A_norm = A / row_sums# Weight matrix (2 features -> 2 output features)W = np.random.randn(2, 2)# One round of message passingaggregated = A_norm @ Houtput = aggregated @ Wprint("Node features H:\n", np.round(H, 3))print("\nAggregated (A_norm @ H):\n", np.round(aggregated, 3))print("\nOutput (aggregated @ W):\n", np.round(output, 3))print("\nNode 0 aggregated features = weighted average of its neighbors (1, 2, 3)")

# Quiz Review

---

Try answering these before reading the explanations:

**Q1:** What does the graph Laplacian L = D − A reveal?
> **Explanation:** Its eigenvalues reveal connected components and connectivity (Fiedler value).

**Q2:** What does BFS find?
> **Explanation:** Shortest paths in unweighted graphs — level by level exploration.

**Q3:** What is one round of message passing?
> **Explanation:** Each node aggregates information from its immediate neighbors — the core of GNNs.



# Practice Exercises

---

Do these in order:

**Exercise 1:** Implement PageRank from scratch on a small web graph.

**Exercise 2:** Build a graph with two clusters connected by a bridge; verify spectral clustering finds the split.



# Key Terms

---

| Term | What people say | What it actually means |
|------|----------------|----------------------|
| Graph | Nodes and edges | G = (V, E) encoding pairwise relationships |
| Adjacency matrix | Connection table | A[i][j] = 1 if nodes i and j are connected |
| Laplacian | L = D − A | Matrix whose eigenvalues reveal graph structure |
| Fiedler value | Algebraic connectivity | Smallest non-zero eigenvalue of L |
| BFS | Level-by-level search | Finds shortest paths in unweighted graphs |
| DFS | Go deep first | Finds connected components, cycles |
| Message passing | Nodes talk to neighbors | Core operation of Graph Neural Networks |


# Further Reading

---

- [Kipf & Welling (2017): Graph Convolutional Networks](https://arxiv.org/abs/1609.02907)
- [Hamilton (2020): Graph Representation Learning](https://dl.acm.org/doi/10.5555/3465045)


# Next Steps

---

When you're ready, continue to Lesson 22: Stochastic Processes.

Your `ai-engineering-from-scratch` learning directory is **completely untouched** — these notes live separately in this folder.
